In [1]:
# 3D Particle Visualization with Rerun
# Run experiment for "jello_trim" video and visualize particles in 3D

import os
import sys
import numpy as np
import jax
import jax.numpy as jnp
from jax.random import key as jkey

# JAX compilation cache setup
cache_dir = os.path.join(os.getcwd(), ".jax_cache")
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
jax.config.update("jax_compilation_cache_dir", cache_dir)
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.experimental.compilation_cache.compilation_cache.set_cache_dir(cache_dir)

# Add workspace root to path to import from main script
workspace_root = os.path.dirname(os.getcwd())  # Go up from plotting_scripts to workspace root
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Import rerun
import rerun as rr

# Import functions from the main tracking script
from dino_tracking_subsampling_dense_eval import (
    process_video,
    NUM_BLOBS,
    NUM_HYPERBLOBS_ORIGINAL,
    FOCAL_LENGTH,
    BLOB_COUNTING_THRESHOLD,
    RANDOM_SEED,
    DAVIS_3D_MOTION_PATH,
    DAVIS_SEGMASKS_PATH,
    DAVIS_RGB_PATH,
    DINO_PATH_TEMPLATE,
    EXPERIMENT_SAVE_DIR
)

print("Setup complete!")


Setup complete!


In [2]:
# Override paths with correct locations for this system
import dino_tracking_subsampling_dense_eval as tracking_module

# Update paths to use cvpr_deformable_experiment directories
tracking_module.DAVIS_3D_MOTION_PATH = "/home/esli/GenMatter/assets/cvpr_deformable_experiment/npzs"
tracking_module.DINO_PATH_TEMPLATE = '/home/esli/GenMatter/assets/cvpr_deformable_experiment/dino_features/dino_features_pca_l/{}_dino_pca_per_pixel.npz'

# Set segmentation masks path
import os
segmask_path = "/home/esli/GenMatter/assets/cvpr_deformable_experiment/segmasks"
if os.path.exists(segmask_path):
    tracking_module.DAVIS_SEGMASKS_PATH = segmask_path
    print(f"Using segmentation masks at: {segmask_path}")
else:
    print(f"Warning: Segmentation mask path not found: {segmask_path}")
# Enable SAM frame0 and set path
tracking_module.USE_SAM_FRAME0 = True
tracking_module.SAM_FRAME0_PATH_TEMPLATE = '/home/esli/GenMatter/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_SAM_frame0/{}_SAM_frame0.png'
print("Enabled SAM frame0 (using SAM segmentation masks)")

tracking_module.DAVIS_RGB_PATH = "/home/esli/GenMatter/assets/cvpr_deformable_experiment/frames"
tracking_module.DAVIS_RGB_PATH = "/home/esli/GenMatter/assets/tapvid_davis_30_videos_processed/tapvid_davis_rgb_frames"
print("Updated paths:")
print(f"  3D Motion: {tracking_module.DAVIS_3D_MOTION_PATH}")
print(f"  DINO Features: {tracking_module.DINO_PATH_TEMPLATE}")
print(f"  Segmentation Masks: {tracking_module.DAVIS_SEGMASKS_PATH}")
print(f"  RGB Frames: {tracking_module.DAVIS_RGB_PATH}")
print(f"  SAM Frame0: {tracking_module.SAM_FRAME0_PATH_TEMPLATE}")

# Run experiment for "jello_trim" video
VIDEO_NAME = "jello_trim" #purple_jacket # "jello_trim"
SUBSAMPLING_PERCENTAGE = 100.0  # Use full resolution

print(f"\nRunning experiment for video: {VIDEO_NAME}")
print(f"Subsampling: {SUBSAMPLING_PERCENTAGE}%")

# Process the video
result = process_video(
    video_name=VIDEO_NAME,
    subsampling_percentage=SUBSAMPLING_PERCENTAGE,
    subsampled_indices=None,
    run_save_dir=None
)

if result is None:
    raise RuntimeError(f"Failed to process video {VIDEO_NAME}")

print(f"\nExperiment complete!")
print(f"Number of frames: {len(result['tracking_data'])}")
print(f"Image dimensions: {result['img_dims']}")


Using segmentation masks at: /home/esli/GenMatter/assets/cvpr_deformable_experiment/segmasks
Enabled SAM frame0 (using SAM segmentation masks)
Updated paths:
  3D Motion: /home/esli/GenMatter/assets/cvpr_deformable_experiment/npzs
  DINO Features: /home/esli/GenMatter/assets/cvpr_deformable_experiment/dino_features/dino_features_pca_l/{}_dino_pca_per_pixel.npz
  Segmentation Masks: /home/esli/GenMatter/assets/cvpr_deformable_experiment/segmasks
  RGB Frames: /home/esli/GenMatter/assets/tapvid_davis_30_videos_processed/tapvid_davis_rgb_frames
  SAM Frame0: /home/esli/GenMatter/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_SAM_frame0/{}_SAM_frame0.png

Running experiment for video: jello_trim
Subsampling: 100.0%

Processing: jello_trim | Subsampling: 100.0%
Limited to first 50 frames for jello_trim
Running initial Gibbs sweeps...
Running tracking...
JIT compiling tracking function...
Measuring FPS after JIT compilation...
Tracking FPS: 1.64 frames/second (total time: 29.86s f

Dense Evaluating Blob Assignments: 100%|██████████| 50/50 [00:10<00:00,  4.60it/s]


Computing error rates...
is_mask_subsampled? ~~~~~~~~~~~> : False

📊 Processing dataset: jello_trim

📈 Trial Results for jello_trim:
Trial    Recall (%)   Precision  FPR      Jaccard  Accuracy   MW-F1-A    MW-F1-F    MW-J-A   MW-J-F   MW-Acc-A   MW-Acc-F   AUC     
--------------------------------------------------------------------------------------------------------------------------------------
1        82.68        0.956      0.006    0.796    0.974      0.983      0.983      0.967    0.966    0.988      0.988      0.009   
--------------------------------------------------------------------------------------------------------------------------------------
Note: MW-F1-A = Matter-Weighted F1 (Adaptive weights), MW-F1-F = Matter-Weighted F1 (Fixed frame-0 weights)
      MW-J-A = Matter-Weighted Jaccard (Adaptive weights), MW-J-F = Matter-Weighted Jaccard (Fixed frame-0 weights)
      MW-Acc-A = Matter-Weighted Accuracy (Adaptive weights), MW-Acc-F = Matter-Weighted Accuracy (Fixed fr

In [3]:
# Extract particle data from tracking results
tracking_data = result['tracking_data']
segmentation_masks = result['segmentation_masks']
img_dims = result['img_dims']

# Extract particle means and covariances for each frame
particle_means = []  # List of (T, N, 3) arrays
particle_covs = []   # List of (T, N, 3, 3) arrays
particle_weights = []  # List of (T, N) arrays
is_object_particle = []  # List of (T, N) boolean arrays

fx = fy = FOCAL_LENGTH
cx = img_dims[1] / 2.0
cy = img_dims[0] / 2.0

# Determine object particles from frame 0 (same logic as compute_error_rates)
frame0 = tracking_data[0]
blob_assignments_frame0 = frame0['blob_assignments']
n_blobs_frame0 = frame0['n_blobs']
gt_mask_frame0 = segmentation_masks[0]

# Count pixels per blob in frame 0
blob_pixel_counts_frame0 = np.bincount(
    blob_assignments_frame0[blob_assignments_frame0 < n_blobs_frame0],
    minlength=n_blobs_frame0
)
significant_blobs = np.where(blob_pixel_counts_frame0 >= BLOB_COUNTING_THRESHOLD)[0]

# Project blob means to 2D and determine object vs background particles
blob_means_frame0 = frame0['blob_means']
x_2d = (blob_means_frame0[:, 0] / (blob_means_frame0[:, 2] + 1e-8)) * fx + cx
y_2d = (blob_means_frame0[:, 1] / (blob_means_frame0[:, 2] + 1e-8)) * fy + cy
x_2d = np.clip(x_2d.astype(int), 0, img_dims[1] - 1)
y_2d = np.clip(y_2d.astype(int), 0, img_dims[0] - 1)

object_blobs_frame0 = []
background_blobs_frame0 = []

for blob_idx in significant_blobs:
    pixel_idx = y_2d[blob_idx] * img_dims[1] + x_2d[blob_idx]
    is_on_mask = pixel_idx < len(gt_mask_frame0) and gt_mask_frame0[pixel_idx]
    if is_on_mask:
        object_blobs_frame0.append(blob_idx)
    else:
        background_blobs_frame0.append(blob_idx)

object_blobs_frame0 = np.array(object_blobs_frame0)
background_blobs_frame0 = np.array(background_blobs_frame0)

print(f"Found {len(object_blobs_frame0)} object particles and {len(background_blobs_frame0)} background particles")

# Extract data for all frames
for frame_idx in range(len(tracking_data)):
    frame = tracking_data[frame_idx]
    blob_means = np.array(frame['blob_means'])  # (N, 3)
    blob_covs = np.array(frame['blob_covs'])    # (N, 3, 3)
    blob_weights = np.array(frame['blob_weights'])  # (N,)
    n_blobs = frame['n_blobs']
    
    # Determine which particles are object particles (based on frame 0 classification)
    # Only consider particles that still exist in current frame
    is_object = np.zeros(n_blobs, dtype=bool)
    for obj_idx in object_blobs_frame0:
        if obj_idx < n_blobs:
            is_object[obj_idx] = True
    
    particle_means.append(blob_means)
    particle_covs.append(blob_covs)
    particle_weights.append(blob_weights)
    is_object_particle.append(is_object)

print(f"Extracted data for {len(particle_means)} frames")
print(f"Particle means shape: {particle_means[0].shape}")
print(f"Particle covs shape: {particle_covs[0].shape}")


Found 53 object particles and 445 background particles
Extracted data for 50 frames
Particle means shape: (498, 3)
Particle covs shape: (498, 3, 3)


In [5]:
# Synchronized Side-by-Side Visualization: RGB Frame, Point Clouds and Particles
# This cell creates a single Rerun visualization with three synchronized viewers:
# - Left: Original RGB frame (2D image)
# - Middle & Right: Two 3D views showing point clouds AND particles (ellipsoids)
# All views are synchronized on the same timeline - scrolling updates all simultaneously

print("Creating synchronized side-by-side visualization with point clouds and particles...")

# Import required libraries
import cv2
from glob import glob
from scipy.linalg import eigh
import matplotlib.cm as cm

# Use the same frame indices as Cell 5 (every 5th frame)
frame_indices_to_visualize = list(range(0, len(tracking_data), 1))  # Every 5th frame
num_frames = len(frame_indices_to_visualize)

# Visualization parameters (same as Cell 5 and Cell 6)
BACKGROUND_DARKENING_FACTOR = 1.0
POINT_SUBSAMPLE_FACTOR = 1
POINT_RADIUS = 0.015
ELLIPSOID_SCALE = 2.0
MAX_PARTICLES_TO_SHOW = 500
SIZE_FILTER_THRESHOLD = 5.0
PRINCIPAL_AXIS_RATIO_THRESHOLD = 4.0
# Weight-based culling: filter out particles with weights below this percentile
# Set to 0 to disable weight-based filtering, higher values (e.g., 10-20) filter more aggressively
MIN_WEIGHT_PERCENTILE = 20.0  # Keep particles above this percentile of weights

# Helper function to create ellipsoid mesh (same as Cell 6)
def create_ellipsoid_mesh(mean, cov, scale=1.0, num_points=32):
    """Create ellipsoid mesh vertices from mean and covariance matrix"""
    eigenvals, eigenvecs = eigh(cov)
    eigenvals = np.maximum(eigenvals, 1e-6)
    radii = np.sqrt(eigenvals) * scale
    
    u = np.linspace(0, 2 * np.pi, num_points)
    v = np.linspace(0, np.pi, num_points)
    u, v = np.meshgrid(u, v)
    
    x = np.sin(v) * np.cos(u)
    y = np.sin(v) * np.sin(u)
    z = np.cos(v)
    
    sphere_points = np.stack([x.flatten(), y.flatten(), z.flatten()], axis=1)
    sphere_points = sphere_points * radii
    sphere_points = sphere_points @ eigenvecs.T
    ellipsoid_points = sphere_points + mean
    
    n_u, n_v = num_points, num_points
    faces = []
    for i in range(n_v - 1):
        for j in range(n_u - 1):
            idx = i * n_u + j
            faces.append([idx, idx + 1, idx + n_u])
            faces.append([idx + 1, idx + n_u + 1, idx + n_u])
    
    return ellipsoid_points, np.array(faces)

# Load RGB frames
print("Loading RGB frames...")
rgb_base_path = "/home/esli/GenMatter/assets/cvpr_deformable_experiment/frames"
rgb_dir = os.path.join(rgb_base_path, VIDEO_NAME)
rgb_files = sorted(glob(os.path.join(rgb_dir, "*.jpg")))
if len(rgb_files) == 0:
    rgb_files = sorted(glob(os.path.join(rgb_dir, "*.png")))

rgb_frames = []
if len(rgb_files) > 0:
    for frame_idx in frame_indices_to_visualize:
        if frame_idx < len(rgb_files):
            frame = cv2.imread(rgb_files[frame_idx])
            if frame is not None:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                rgb_frames.append(frame)
    print(f"Loaded {len(rgb_frames)} RGB frames")
else:
    print(f"Warning: No RGB frames found in {rgb_dir}")

# Compute average RGB colors per particle (needed for particle visualization)
print("Computing average RGB colors per particle...")
particle_rgb_colors = []  # List of (N, 3) arrays with RGB colors

for vis_idx, frame_idx in enumerate(frame_indices_to_visualize):
    frame_data = tracking_data[frame_idx]
    blob_assignments = frame_data['blob_assignments']
    n_blobs = frame_data['n_blobs']
    
    # Reshape assignments to image dimensions
    blob_assignments_2d = blob_assignments.reshape(img_dims)
    
    # Get RGB frame
    if vis_idx < len(rgb_frames):
        rgb_frame = rgb_frames[vis_idx]
        # Resize if needed
        if rgb_frame.shape[:2] != img_dims:
            rgb_frame = cv2.resize(rgb_frame, (img_dims[1], img_dims[0]), interpolation=cv2.INTER_LINEAR)
    else:
        # Use default gray color if no RGB frame
        rgb_frame = np.full((img_dims[0], img_dims[1], 3), 128, dtype=np.uint8)
    
    # Compute average color per blob
    blob_colors = np.zeros((n_blobs, 3), dtype=np.float32)
    for blob_idx in range(n_blobs):
        mask = (blob_assignments_2d == blob_idx)
        if np.any(mask):
            blob_colors[blob_idx] = np.mean(rgb_frame[mask], axis=0)
        else:
            blob_colors[blob_idx] = [128, 128, 128]  # Default gray
    
    particle_rgb_colors.append(blob_colors.astype(np.uint8))

print(f"Computed average RGB colors for {len(particle_rgb_colors)} frames")

# Initialize rerun with a single session
rr.init("synchronized_point_clouds_and_particles", spawn=True)

# Set up coordinate system
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_UP)

# Set up Blueprint with two side-by-side 3D views BEFORE logging data
try:
    import rerun.blueprint as rrb
    
    # Create two 3D views side by side using Horizontal layout
    # Note: Images are logged to "world/rgb_frame" and will be visible in the entity tree
    # You can manually add an image view in the Rerun UI by right-clicking "world/rgb_frame" > "Add View"
    # All views (including manually added image view) will be synchronized on the same timeline
    blueprint = rrb.Blueprint(
        rrb.Horizontal(
            rrb.Spatial3DView(
                origin="/world",
                name="3D View 1",
                background=[255, 255, 255],
            ),
            rrb.Spatial3DView(
                origin="/world",
                name="3D View 2",
                background=[255, 255, 255],
            ),
        ),
        collapse_panels=True,
    )
    rr.send_blueprint(blueprint)
    print("Set up two synchronized 3D views side by side")
    print("RGB frames are logged to 'world/rgb_frame' - add an image view manually in Rerun UI:")
    print("  Right-click 'world/rgb_frame' in the entity tree > 'Add View' > 'Image'")
    print("All views will be synchronized on the same timeline")
except Exception as e:
    print(f"Warning: Could not set up side-by-side views automatically: {e}")
    print("You may need to manually arrange views in the Rerun viewer")

print(f"\nVisualizing {num_frames} frames (every 5th frame) with synchronized point clouds and particles")
print(f"Frame indices: {frame_indices_to_visualize}")

# Visualize each frame
for vis_idx, frame_idx in enumerate(frame_indices_to_visualize):
    frame_data = tracking_data[frame_idx]
    
    # Set timeline (shared by all views)
    rr.set_time_sequence("frame", vis_idx)
    
    # ===== RGB FRAME VISUALIZATION (Left View) =====
    # Log the original RGB frame as an image
    if vis_idx < len(rgb_frames):
        rgb_frame = rgb_frames[vis_idx]
        if rgb_frame.shape[:2] != img_dims:
            rgb_frame = cv2.resize(rgb_frame, (img_dims[1], img_dims[0]), interpolation=cv2.INTER_LINEAR)
        # Ensure image is in the correct format (HWC, uint8)
        if rgb_frame.dtype != np.uint8:
            rgb_frame = rgb_frame.astype(np.uint8)
        # Log RGB frame (visible in left view)
        rr.log(
            "world/rgb_frame",
            rr.Image(rgb_frame)
        )
    
    # ===== POINT CLOUD VISUALIZATION (Middle & Right Views) =====
    # Get 3D positions and assignments
    datapoint_positions = frame_data['datapoint_positions']
    blob_assignments = frame_data['blob_assignments']
    n_blobs = frame_data['n_blobs']
    
    # Filter out outlier assignments
    valid_mask = blob_assignments < n_blobs
    valid_positions = datapoint_positions[valid_mask]
    valid_assignments = blob_assignments[valid_mask]
    
    # Subsample points if needed
    if POINT_SUBSAMPLE_FACTOR > 1:
        subsample_indices = np.arange(0, len(valid_positions), POINT_SUBSAMPLE_FACTOR)
        valid_positions = valid_positions[subsample_indices]
        valid_assignments = valid_assignments[subsample_indices]
    
    # Flip x and y coordinates to match Cell 5
    valid_positions_flipped = valid_positions.copy()
    valid_positions_flipped[:, 0] = -valid_positions_flipped[:, 0]
    valid_positions_flipped[:, 1] = -valid_positions_flipped[:, 1]
    
    # Get object particle mask
    is_object = is_object_particle[frame_idx]
    is_object_pixel = is_object[valid_assignments]
    
    # Get original RGB colors for point cloud (project 3D points back to 2D to get colors)
    if vis_idx < len(rgb_frames):
        rgb_frame_for_colors = rgb_frames[vis_idx]
        if rgb_frame_for_colors.shape[:2] != img_dims:
            rgb_frame_for_colors = cv2.resize(rgb_frame_for_colors, (img_dims[1], img_dims[0]), interpolation=cv2.INTER_LINEAR)
        
        fx = fy = FOCAL_LENGTH
        cx = img_dims[1] / 2.0
        cy = img_dims[0] / 2.0
        
        x_2d = (valid_positions[:, 0] / (valid_positions[:, 2] + 1e-8)) * fx + cx
        y_2d = (valid_positions[:, 1] / (valid_positions[:, 2] + 1e-8)) * fy + cy
        x_2d = np.clip(x_2d.astype(int), 0, img_dims[1] - 1)
        y_2d = np.clip(y_2d.astype(int), 0, img_dims[0] - 1)
        
        original_colors = rgb_frame_for_colors[y_2d, x_2d]
    else:
        original_colors = np.full((len(valid_positions), 3), 128, dtype=np.uint8)
    
    # Convert to RGBA
    original_colors_rgba = np.zeros((len(valid_positions), 4), dtype=np.uint8)
    original_colors_rgba[:, :3] = original_colors
    original_colors_rgba[:, 3] = 255
    
    # Log point cloud (visible in both 3D views)
    rr.log(
        "world/point_clouds/original_rgb",
        rr.Points3D(
            positions=valid_positions_flipped,
            colors=original_colors_rgba,
            radii=POINT_RADIUS
        )
    )
    
    # ===== PARTICLE VISUALIZATION (visible in both 3D views) =====
    # Get particle data for this frame
    means = particle_means[frame_idx].copy()
    covs = particle_covs[frame_idx].copy()
    weights = particle_weights[frame_idx].copy()
    is_object_particles = is_object_particle[frame_idx].copy()
    n_particles_original = len(means)
    n_particles = n_particles_original
    
    # Apply weight-based culling to reduce flicker from low-weight particles
    weight_mask = None
    if MIN_WEIGHT_PERCENTILE > 0 and n_particles > 0:
        weight_threshold = np.percentile(weights, MIN_WEIGHT_PERCENTILE)
        weight_mask = weights >= weight_threshold
        n_filtered = np.sum(~weight_mask)
        if n_filtered > 0:
            means = means[weight_mask]
            covs = covs[weight_mask]
            weights = weights[weight_mask]
            is_object_particles = is_object_particles[weight_mask]
            n_particles = len(means)
            if vis_idx == 0 or vis_idx % 10 == 0:  # Print every 10th frame to avoid spam
                print(f"Frame {frame_idx}: Weight-based culling filtered out {n_filtered} particles (kept {n_particles} above {MIN_WEIGHT_PERCENTILE}th percentile, threshold={weight_threshold:.6f})")
    
    # Apply coordinate flip to match point clouds
    flip_matrix = np.array([[-1, 0, 0], [0, -1, 0], [0, 0, 1]], dtype=np.float32)
    means[:, 0] = -means[:, 0]
    means[:, 1] = -means[:, 1]
    for i in range(len(covs)):
        covs[i] = flip_matrix @ covs[i] @ flip_matrix.T
    
    # Get colors for particles (RGB colors) - get full array first
    if vis_idx < len(particle_rgb_colors):
        rgb_colors_full = particle_rgb_colors[vis_idx].copy()
    else:
        rgb_colors_full = np.full((n_particles_original, 3), 128, dtype=np.uint8)
    
    # Get cluster colors for particles - get full array first
    frame_data_cluster = tracking_data[frame_idx]
    hyperblob_assignments_frame = frame_data_cluster['hyperblob_assignments']  # (n_blobs,) - maps particles to clusters
    n_hyperblobs = frame_data_cluster['n_hyperblobs']
    
    # Generate distinct colors for each cluster using the same colormap as the other notebook
    cluster_colormap = cm.get_cmap('tab20')  # 20 distinct colors, will cycle if more clusters
    cluster_colors_rgb = np.zeros((n_hyperblobs + 1, 3), dtype=np.uint8)  # +1 for outlier cluster
    for cluster_idx in range(n_hyperblobs + 1):
        # Use modulo to cycle through colormap if there are more clusters than colors
        color_rgba = cluster_colormap(cluster_idx % 20)
        cluster_colors_rgb[cluster_idx] = (np.array(color_rgba[:3]) * 255).astype(np.uint8)
    
    # Map each particle to its cluster color (full array)
    cluster_colors_full = cluster_colors_rgb[hyperblob_assignments_frame]  # (n_blobs, 3)
    
    # Apply weight mask to colors if weight filtering was applied
    if weight_mask is not None:
        rgb_colors = rgb_colors_full[weight_mask]
        cluster_colors = cluster_colors_full[weight_mask]
    else:
        rgb_colors = rgb_colors_full
        cluster_colors = cluster_colors_full
    
    # Limit particles if too many
    if n_particles > MAX_PARTICLES_TO_SHOW:
        indices = np.argsort(weights)[-MAX_PARTICLES_TO_SHOW:]
        means = means[indices].copy()
        covs = covs[indices].copy()
        weights = weights[indices].copy()
        rgb_colors = rgb_colors[indices].copy()
        cluster_colors = cluster_colors[indices].copy()
        is_object_particles = is_object_particles[indices].copy()
        n_particles = MAX_PARTICLES_TO_SHOW
    
    # Filter particles by size and shape (same logic as Cell 6)
    volumes = np.array([np.sqrt(np.linalg.det(cov)) for cov in covs])
    reference_volume = np.percentile(volumes, 75)
    
    valid_particle_mask = np.ones(n_particles, dtype=bool)
    
    if SIZE_FILTER_THRESHOLD > 0:
        size_filtered = volumes > (SIZE_FILTER_THRESHOLD * reference_volume)
        valid_particle_mask = valid_particle_mask & ~size_filtered
    
    if PRINCIPAL_AXIS_RATIO_THRESHOLD > 0:
        for i in range(n_particles):
            cov = covs[i]
            eigenvals, eigenvecs = eigh(cov)
            eigenvals = np.maximum(eigenvals, 1e-10)
            eigenvals_sorted = np.sort(eigenvals)[::-1]
            first_principal_axis = np.sqrt(eigenvals_sorted[0])
            second_principal_axis = np.sqrt(eigenvals_sorted[1])
            if second_principal_axis > 1e-6:
                axis_ratio = first_principal_axis / second_principal_axis
                if axis_ratio >= PRINCIPAL_AXIS_RATIO_THRESHOLD:
                    valid_particle_mask[i] = False
    
    # Apply filter
    means = means[valid_particle_mask]
    covs = covs[valid_particle_mask]
    weights = weights[valid_particle_mask]
    rgb_colors = rgb_colors[valid_particle_mask]
    cluster_colors = cluster_colors[valid_particle_mask]
    is_object_particles = is_object_particles[valid_particle_mask]
    n_particles = len(means)
    
    # Compute opacity based on volume
    volumes = np.array([np.sqrt(np.linalg.det(cov)) for cov in covs])
    if volumes.max() > volumes.min():
        normalized_volumes = (volumes - volumes.min()) / (volumes.max() - volumes.min())
    else:
        normalized_volumes = np.ones(n_particles)
    opacity_from_volume = 1.0 - normalized_volumes * 0.75
    opacity_from_volume = np.clip(opacity_from_volume, 0.25, 1.0)
    opacity_values = opacity_from_volume
    
    # Visualize particles as ellipsoids (right view) - using RGB colors
    for i in range(n_particles):
        mean = means[i]
        cov = covs[i]
        color_rgb = rgb_colors[i]
        opacity = opacity_values[i]
        
        vertices, faces = create_ellipsoid_mesh(mean, cov, scale=ELLIPSOID_SCALE)
        num_vertices = len(vertices)
        
        alpha_uint8 = int(opacity * 255)
        vertex_colors = np.zeros((num_vertices, 4), dtype=np.uint8)
        vertex_colors[:, :3] = color_rgb
        vertex_colors[:, 3] = alpha_uint8
        
        # Log ellipsoid colored by RGB (visible in both 3D views)
        rr.log(
            f"world/rgb_particles/ellipsoid_{i}",
            rr.Mesh3D(
                vertex_positions=vertices,
                triangle_indices=faces,
                vertex_colors=vertex_colors
            )
        )
    
    # Visualize particles as ellipsoids colored by cluster assignments (visible in both 3D views)
    for i in range(n_particles):
        mean = means[i]
        cov = covs[i]
        color_rgb = cluster_colors[i]  # RGB color from cluster assignment
        opacity = opacity_values[i]  # Opacity inversely proportional to volume
        
        vertices, faces = create_ellipsoid_mesh(mean, cov, scale=ELLIPSOID_SCALE)
        num_vertices = len(vertices)
        
        alpha_uint8 = int(opacity * 255)
        vertex_colors = np.zeros((num_vertices, 4), dtype=np.uint8)
        vertex_colors[:, :3] = color_rgb
        vertex_colors[:, 3] = alpha_uint8
        
        # Log ellipsoid colored by cluster (visible in both 3D views)
        rr.log(
            f"world/cluster_particles/ellipsoid_{i}",
            rr.Mesh3D(
                vertex_positions=vertices,
                triangle_indices=faces,
                vertex_colors=vertex_colors
            )
        )

print("\n" + "="*60)
print("Synchronized visualization ready! Check the Rerun viewer.")
print("="*60)
print("\nTips:")
print("  - Three synchronized views are shown side by side:")
print("    Left: Original RGB frame (2D image)")
print("    Middle & Right: Two 3D views showing the SAME data")
print("  - All three views are synchronized on the same timeline")
print("  - Use the timeline slider at the bottom to scrub through frames - all views update together")
print("  - The 3D views show point clouds AND particles (ellipsoids)")
print("  - Use mouse to rotate/pan/zoom the 3D views independently")
print("  - Point clouds show all 3D pixels colored by their original RGB values")
print("  - Two sets of particles are shown:")
print("    1. rgb_particles: Particles colored by average RGB colors from video frames")
print("    2. cluster_particles: Particles colored by their cluster/hyperblob assignment")
print("  - Ellipsoid opacity is inversely proportional to volume")
print("  - Weight-based culling filters out low-weight particles to reduce flicker")
print(f"    (MIN_WEIGHT_PERCENTILE={MIN_WEIGHT_PERCENTILE} - adjust this parameter to control filtering)")
print("  - Toggle visibility of each particle set in Rerun viewer to compare them")
print("="*60)


Creating synchronized side-by-side visualization with point clouds and particles...
Loading RGB frames...
Loaded 50 RGB frames
Computing average RGB colors per particle...
Computed average RGB colors for 50 frames
Set up two synchronized 3D views side by side
RGB frames are logged to 'world/rgb_frame' - add an image view manually in Rerun UI:
  Right-click 'world/rgb_frame' in the entity tree > 'Add View' > 'Image'
All views will be synchronized on the same timeline

Visualizing 50 frames (every 5th frame) with synchronized point clouds and particles
Frame indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Frame 0: Weight-based culling filtered out 100 particles (kept 398 above 20.0th percentile, threshold=0.000803)


/var/tmp/ipykernel_175413/4135035405.py:162: DeprecationWarning: Use `set_time(sequence=…)` instead.
    See: https://www.rerun.io/docs/reference/migration/migration-0-23 for more details.
  rr.set_time_sequence("frame", vis_idx)
/var/tmp/ipykernel_175413/4135035405.py:282: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cluster_colormap = cm.get_cmap('tab20')  # 20 distinct colors, will cycle if more clusters


Frame 10: Weight-based culling filtered out 100 particles (kept 398 above 20.0th percentile, threshold=0.000802)
Frame 20: Weight-based culling filtered out 100 particles (kept 398 above 20.0th percentile, threshold=0.000849)
Frame 30: Weight-based culling filtered out 100 particles (kept 398 above 20.0th percentile, threshold=0.000728)
Frame 40: Weight-based culling filtered out 100 particles (kept 398 above 20.0th percentile, threshold=0.000723)

Synchronized visualization ready! Check the Rerun viewer.

Tips:
  - Three synchronized views are shown side by side:
    Left: Original RGB frame (2D image)
    Middle & Right: Two 3D views showing the SAME data
  - All three views are synchronized on the same timeline
  - Use the timeline slider at the bottom to scrub through frames - all views update together
  - The 3D views show point clouds AND particles (ellipsoids)
  - Use mouse to rotate/pan/zoom the 3D views independently
  - Point clouds show all 3D pixels colored by their origina